## Exercise 16: A complete ML pipeline for modeling diamond prices (`diamonds.csv`)

**Task (from the book's exercise sheet):** "In this exercise you will build a complete ML
pipeline in which you model diamond prices, available in the dataset 'diamonds.csv'."
(Dataset reference: https://www.kaggle.com/datasets/shivam2503/diamonds)

Below is a complete regression ML pipeline, following the same checklist/structure
introduced in Chapter 2 (EDA, data processing, train/val/test split, training and
comparing several of the regression models covered in this chapter, and final evaluation
on the test set), using several of the models from Question 6 (linear regression, ridge,
lasso and random forest).


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score


diamonds = pd.read_csv("diamonds .csv")
diamonds = diamonds.drop(columns=["Unnamed: 0"])  # plain row index, no information content
print("Shape:", diamonds.shape)
diamonds.head()


Shape: (53940, 10)


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [6]:
diamonds.info()
print("\nMissing values:\n", diamonds.isna().sum())
print("\nDuplicates:", diamonds.duplicated().sum())
diamonds.describe()


<class 'pandas.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  str    
 2   color    53940 non-null  str    
 3   clarity  53940 non-null  str    
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB

Missing values:
 carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64

Duplicates: 146


,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


**The data** contains 53,940 diamonds with `carat` (weight), three categorical
quality variables (`cut`, `color`, `clarity`), `depth`/`table` (proportions in percent),
physical dimensions `x`, `y`, `z` (mm), and `price` (the target variable, USD).

**Data quality issue:** `x`, `y`, `z` describe the diamond's physical dimensions in
millimeters and can therefore never reasonably be 0. We check for, and clean out, such
invalid observations.

In [7]:
bad_dims = (diamonds[["x", "y", "z"]] == 0).any(axis=1)
print("Number of rows with a zero dimension (x/y/z):", bad_dims.sum())
diamonds[bad_dims]


Number of rows with a zero dimension (x/y/z): 20


,carat,cut,color,clarity,depth,table,price,x,y,z
2207,1.00,Premium,G,SI2,59.1,59.0,3142,6.55,6.48,0.0
2314,1.01,Premium,H,I1,58.1,59.0,3167,6.66,6.60,0.0
4791,1.10,Premium,G,SI2,63.0,59.0,3696,6.50,6.47,0.0
5471,1.01,Premium,F,SI2,59.2,58.0,3837,6.50,6.47,0.0
10167,1.50,Good,G,I1,64.0,61.0,4731,7.15,7.04,0.0
11182,1.07,Ideal,F,SI2,61.6,56.0,4954,0.00,6.62,0.0
11963,1.00,Very Good,H,VS2,63.3,53.0,5139,0.00,0.00,0.0
13601,1.15,Ideal,G,VS2,59.2,56.0,5564,6.88,6.83,0.0
15951,1.14,Fair,G,VS1,57.5,67.0,6381,0.00,0.00,0.0
24394,2.18,Premium,H,SI2,59.4,61.0,12631,8.49,8.45,0.0


In [8]:
diamonds = diamonds[~bad_dims].reset_index(drop=True)
print("Shape after cleaning:", diamonds.shape)


Shape after cleaning: (53920, 10)
